# DreamBooth Rich-Prompt Notebook — Dog2 + Cat2 + Clock, LR 5e-6, Full Step-400 Preview

This notebook is a focused qualitative DreamBooth run using **dog2**, **cat2**, and **clock**.

It keeps the same structure as the earlier focused notebook, including:
- prior vs no-prior runs
- inline image viewing in Colab
- a **full 8-prompt × 4-image preview at training step 400**
- a final full **8-prompt × 4-image grid at training step 800**
- optional headline-subject ablations

This version uses **lr_unet = 5e-6** and writes to a **new output folder** so it does **not overwrite final1** or the earlier final2 notebooks.

## 1. Install + Imports + Drive Mount

In [25]:
# Install dependencies (run once per session)
!pip -q install --upgrade diffusers transformers accelerate safetensors lpips ftfy timm
import torch
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch 2.10.0+cu128, CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [26]:
import os
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
else:
    print("Drive already mounted.")


Drive already mounted.


In [27]:
import gc, json, math, csv, pickle, random, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from diffusers import (
    AutoencoderKL,
    UNet2DConditionModel,
    DDPMScheduler,
    StableDiffusionPipeline,
    DPMSolverMultistepScheduler,
)
from transformers import CLIPTokenizer, CLIPTextModel, CLIPModel, CLIPProcessor
import lpips

warnings.filterwarnings("ignore")


## 2. Configuration

In [28]:
CFG = {
    # Paths
    "base_path": "/content/drive/MyDrive/FinalProjectDL",
    "output_path": "/content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview",
    "dataset_root": "/content/drive/MyDrive/FinalProjectDL/dreambooth_dataset/dataset",
    "class_images_dirname": "class_images_final",

    # Model
    "pretrained_model": "runwayml/stable-diffusion-v1-5",
    "identifier": "sks",
    "seed": 42,

    # Focused subjects: two living subjects + one object
    "subjects": [
        {"name": "dog2",  "class_noun": "dog",   "is_living": True},
        {"name": "cat2",  "class_noun": "cat",   "is_living": True},
        {"name": "clock", "class_noun": "clock", "is_living": False},
    ],
    "headline_subject": "dog2",

    "RUN_DOG_ABLATIONS": False,

    # Prompt templates
    "instance_prompt_template": "a photo of a {identifier} {class_noun}",
    "class_prompt_template": "a photo of a {class_noun}",

    # Class prior generation
    "num_class_images": 200,
    "class_gen_steps": 50,
    "class_gen_batch": 2,

    # Training
    "resolution": 512,
    "train_batch_size": 1,
    "max_train_steps": 800,
    "lr_unet": 2.5e-6,
    "lr_te": 2e-6,
    "lambda_prior": 1.0,
    "max_grad_norm": 1.0,

    # Mid-training preview generation
    "preview_train_steps": [400],
    "preview_num_prompts": 8,
    "preview_images_per_prompt": 4,
    "preview_sample_steps": 50,

    # Final inference
    "sample_steps": 50,
    "guidance_scale": 7.5,
    "images_per_prompt": 4,

    # Metrics
    "clip_model_name": "openai/clip-vit-base-patch32",
    "dino_repo": "facebookresearch/dino:main",
    "dino_model": "dino_vits16",

    # Final runtime release
    "RUN_FINAL_UNASSIGN": False,
}

## 3. Dataset Check

In [29]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def list_images(folder):
    folder = Path(folder)
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS)

def read_class_map(prompts_file):
    """Parse prompts_and_classes.txt into {subject_name: class_noun}."""
    class_map = {}
    in_classes = False
    with open(prompts_file, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("subject_name,class"):
                in_classes = True
                continue
            if in_classes and "," in line:
                parts = line.split(",", 1)
                if len(parts) == 2:
                    class_map[parts[0].strip()] = parts[1].strip()
            elif in_classes and not line:
                break
    return class_map

# Verify dataset
assert DATASET_ROOT.exists(), f"Dataset not found at {DATASET_ROOT}"
CLASS_MAP = read_class_map(DATASET_ROOT / "prompts_and_classes.txt")

for subj in CFG["subjects"]:
    name = subj["name"]
    imgs = list_images(DATASET_ROOT / name)
    expected_class = CLASS_MAP.get(name, "???")
    print(f"  {name:25s} -> {len(imgs)} images, "
          f"class: '{subj['class_noun']}' (file says '{expected_class}')")
print("\n✓ Dataset OK")


  dog2                      -> 6 images, class: 'dog' (file says 'dog')
  cat2                      -> 5 images, class: 'cat' (file says 'cat')
  clock                     -> 6 images, class: 'clock' (file says 'clock')

✓ Dataset OK


## 4. Helper Functions

In [30]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def safe_name(text):
    return text.replace(" ", "_").replace("/", "_")

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def instance_prompt(class_noun, identifier="sks"):
    return f"a photo of a {identifier} {class_noun}"

def class_prompt(class_noun):
    return f"a photo of a {class_noun}"

def save_all_results():
    with open(RESULTS_PKL, "wb") as f:
        pickle.dump(ALL_RESULTS, f)


In [31]:
DOG_PROMPTS = [
    "a photo of a {identifier} dog swimming underwater",
    "a photo of a {identifier} dog sleeping in a doghouse",
    "a photo of a {identifier} dog getting a haircut",
    "a photo of a {identifier} dog in the Acropolis",
    "a photo of a {identifier} dog riding on a skateboard",
    "a photo of a {identifier} dog at the beach wearing sunglasses",
    "a watercolor painting of a {identifier} dog",
    "a photo of a {identifier} dog in a bucket",
]

CAT_PROMPTS = [
    "a photo of a {identifier} cat sleeping on a windowsill",
    "a photo of a {identifier} cat playing with a ball of yarn",
    "a photo of a {identifier} cat inside a cardboard box",
    "a photo of a {identifier} cat wearing a bow tie",
    "a photo of a {identifier} cat in Times Square",
    "a photo of a {identifier} cat on top of a bookshelf",
    "a watercolor painting of a {identifier} cat",
    "a photo of a {identifier} cat wearing a santa hat",
]

CLOCK_PROMPTS = [
    "a photo of a {identifier} clock on a wooden desk",
    "a photo of a {identifier} clock in Times Square",
    "a photo of a {identifier} clock on top of a mountain",
    "a photo of a {identifier} clock in a cozy living room",
    "a photo of a {identifier} clock with a city in the background",
    "a watercolor painting of a {identifier} clock",
    "a painting of a {identifier} clock in the style of Van Gogh",
    "a photo of a {identifier} clock in a bucket",
]

def get_eval_prompts(subject_name, class_noun, is_living, identifier="sks"):
    if subject_name.startswith("dog"):
        return [p.format(identifier=identifier) for p in DOG_PROMPTS]
    if subject_name.startswith("cat"):
        return [p.format(identifier=identifier) for p in CAT_PROMPTS]
    if subject_name == "clock":
        return [p.format(identifier=identifier) for p in CLOCK_PROMPTS]

    ending = "wearing a santa hat" if is_living else "in a bucket"
    return [
        f"a photo of a {identifier} {class_noun} in the snow",
        f"a photo of a {identifier} {class_noun} on the beach",
        f"a photo of a {identifier} {class_noun} in Times Square",
        f"a photo of a {identifier} {class_noun} on top of a mountain",
        f"a photo of a {identifier} {class_noun} with a city in the background",
        f"a watercolor painting of a {identifier} {class_noun}",
        f"a painting of a {identifier} {class_noun} in the style of Van Gogh",
        f"a photo of a {identifier} {class_noun} {ending}",
    ]

def get_font(size=18):
    try:
        return ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", size)
    except Exception:
        return ImageFont.load_default()

def make_labeled_grid(image_rows, row_labels, save_path, title=None, cell_size=256):
    """Build a labeled grid image: len(image_rows) rows x len(image_rows[0]) cols."""
    n_rows = len(image_rows)
    n_cols = max(len(r) for r in image_rows)
    label_w = 360
    grid_w = label_w + n_cols * cell_size
    grid_h = n_rows * cell_size
    grid = Image.new("RGB", (grid_w, grid_h), (255, 255, 255))
    draw = ImageDraw.Draw(grid)
    font = get_font(11)

    for r, (imgs, label) in enumerate(zip(image_rows, row_labels)):
        y = r * cell_size
        draw.text((5, y + cell_size // 3), label[:70], fill=(0, 0, 0), font=font)
        for c, img in enumerate(imgs):
            if isinstance(img, (str, Path)):
                img = Image.open(img).convert("RGB")
            img = img.resize((cell_size, cell_size), Image.BICUBIC)
            grid.paste(img, (label_w + c * cell_size, y))

    grid.save(save_path)
    return save_path

def stitch_existing_grids(grid_paths, labels, save_path, resize_to=None):
    grids = []
    for p in grid_paths:
        img = Image.open(p).convert("RGB")
        if resize_to is not None:
            img = img.resize(resize_to, Image.BICUBIC)
        grids.append(img)

    max_w = max(img.width for img in grids)
    label_h = 28
    total_h = sum(img.height + label_h for img in grids)
    canvas = Image.new("RGB", (max_w, total_h), (255, 255, 255))
    draw = ImageDraw.Draw(canvas)
    font = get_font(16)

    y = 0
    for img, label in zip(grids, labels):
        draw.text((10, y + 4), label, fill=(0, 0, 0), font=font)
        y += label_h
        canvas.paste(img, (0, y))
        y += img.height

    canvas.save(save_path)
    return save_path

print("✓ Dog2/cat2/clock rich prompt banks + grid helpers defined")


✓ Dog2/cat2/clock rich prompt banks + grid helpers defined


In [32]:
class DreamBoothDataset(Dataset):
    """
    DreamBooth training dataset.
    instance_sources: list of (folder_path, prompt_string) pairs.
    Supports both single-subject and multi-subject mixing natively.
    """
    def __init__(self, instance_sources, tokenizer, resolution,
                 class_dir=None, class_prompt_text=None,
                 with_prior=True, n_images=None):
        self.tokenizer = tokenizer
        self.with_prior = with_prior
        self.class_prompt_text = class_prompt_text

        # Collect all instance images with their prompts
        self.instance_items = []
        for folder, prompt_text in instance_sources:
            imgs = list_images(folder)
            if n_images is not None:
                imgs = imgs[:n_images]
            for img_path in imgs:
                self.instance_items.append((img_path, prompt_text))

        if len(self.instance_items) == 0:
            raise ValueError("No instance images found")

        self.class_images = []
        if with_prior:
            self.class_images = list_images(class_dir)
            if len(self.class_images) == 0:
                raise ValueError(f"No class images in {class_dir}")

        self.transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        if self.with_prior:
            return max(len(self.instance_items), len(self.class_images))
        return len(self.instance_items)

    def tokenize(self, prompt):
        return self.tokenizer(
            prompt, padding="max_length", truncation=True,
            max_length=self.tokenizer.model_max_length, return_tensors="pt",
        ).input_ids[0]

    def read_image(self, path):
        return self.transform(Image.open(path).convert("RGB"))

    def __getitem__(self, idx):
        inst_path, inst_prompt = self.instance_items[idx % len(self.instance_items)]
        item = {
            "instance_pixel_values": self.read_image(inst_path),
            "instance_input_ids": self.tokenize(inst_prompt),
        }
        if self.with_prior:
            cls_path = self.class_images[idx % len(self.class_images)]
            item["class_pixel_values"] = self.read_image(cls_path)
            item["class_input_ids"] = self.tokenize(self.class_prompt_text)
        return item

print("✓ DreamBoothDataset defined (handles single + multi-source natively)")


✓ DreamBoothDataset defined (handles single + multi-source natively)


In [33]:
def load_training_components(train_text_encoder=False):
    """Load all model components for training. Returns on CUDA, float32."""
    model_id = CFG["pretrained_model"]
    device = get_device()

    tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
    text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device)
    vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to(device)
    unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet").to(device)
    noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

    vae.requires_grad_(False)
    vae.eval()

    if train_text_encoder:
        text_encoder.requires_grad_(True)
    else:
        text_encoder.requires_grad_(False)
        text_encoder.eval()

    unet.requires_grad_(True)
    return tokenizer, text_encoder, vae, unet, noise_scheduler


def build_inference_pipe(tokenizer, text_encoder, vae, unet, use_fp16=True):
    """Build inference pipeline from trained components (no re-download).

    Important: use_fp16=False during mid-training previews so the shared
    training modules are not accidentally converted to half precision.
    """
    pipe = StableDiffusionPipeline(
        vae=vae,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        unet=unet,
        scheduler=DPMSolverMultistepScheduler.from_pretrained(
            CFG["pretrained_model"], subfolder="scheduler"
        ),
        safety_checker=None,
        feature_extractor=None,
        requires_safety_checker=False,
    )
    if use_fp16:
        pipe.unet.half()
        pipe.text_encoder.half()
        pipe.vae.half()
    pipe = pipe.to(get_device())
    pipe.set_progress_bar_config(disable=True)
    return pipe

print("✓ Training component loaders defined")


✓ Training component loaders defined


In [34]:
@torch.no_grad()
def save_training_preview(pipe, run_dir, subject_name, class_noun, is_living, train_step):
    """Save a small qualitative preview during training, before the model is discarded."""
    run_dir = Path(run_dir)
    preview_dir = run_dir / f"preview_step_{train_step}"
    preview_dir.mkdir(parents=True, exist_ok=True)

    prompts = get_eval_prompts(subject_name, class_noun, is_living, CFG["identifier"])
    prompts = prompts[:CFG["preview_num_prompts"]]
    generator = torch.Generator(device=get_device()).manual_seed(CFG["seed"] + train_step)

    image_rows = []
    saved_rows = []
    print(f"\nGenerating mid-training preview at training step {train_step}...")
    for p_idx, prompt in enumerate(tqdm(prompts, desc=f"preview step {train_step}")):
        row_imgs = []
        row_paths = []
        for i_idx in range(CFG["preview_images_per_prompt"]):
            img = pipe(
                prompt,
                num_inference_steps=CFG["preview_sample_steps"],
                guidance_scale=CFG["guidance_scale"],
                generator=generator,
            ).images[0]
            path = preview_dir / f"preview_p{p_idx:02d}_i{i_idx}.png"
            img.save(path)
            row_imgs.append(img)
            row_paths.append(str(path))
        image_rows.append(row_imgs)
        saved_rows.append(row_paths)

    grid_path = run_dir / f"preview_step_{train_step}.png"
    make_labeled_grid(
        image_rows=image_rows,
        row_labels=prompts,
        save_path=grid_path,
        title=f"{subject_name}/{run_dir.name} preview step {train_step}",
    )
    print("Saved preview grid:", grid_path)
    return str(grid_path), saved_rows

print("✓ Mid-training preview helper defined")


✓ Mid-training preview helper defined


In [35]:
def train_dreambooth(run_dir, instance_sources, class_noun, with_prior=True,
                     train_text_encoder=False, n_images=None, max_steps=None):
    """
    Train DreamBooth. Returns inference pipeline directly (no CPU state dict copy).
    Concatenates instance + class into one batch for a single forward pass.
    Saves small preview grids at CFG["preview_train_steps"] during training.
    """
    seed_everything(CFG["seed"])
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    max_steps = max_steps or CFG["max_train_steps"]
    lr = CFG["lr_te"] if train_text_encoder else CFG["lr_unet"]
    device = get_device()

    subject_name = run_dir.parent.name
    is_living = class_noun in {"dog", "cat", "person"}
    preview_steps = set(CFG.get("preview_train_steps", []))

    tokenizer, text_encoder, vae, unet, noise_scheduler = load_training_components(train_text_encoder)

    class_dir = CLASS_ROOT / safe_name(class_noun)
    dataset = DreamBoothDataset(
        instance_sources=instance_sources,
        tokenizer=tokenizer,
        resolution=CFG["resolution"],
        class_dir=class_dir,
        class_prompt_text=class_prompt(class_noun),
        with_prior=with_prior,
        n_images=n_images,
    )
    loader = DataLoader(dataset, batch_size=CFG["train_batch_size"], shuffle=True, num_workers=0)

    params = list(unet.parameters())
    if train_text_encoder:
        params += list(text_encoder.parameters())
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-2)

    loss_rows = []
    global_step = 0
    unet.train()
    if train_text_encoder:
        text_encoder.train()

    print(f"\n{'='*60}")
    print(f"Training: {run_dir.name}")
    print(f"  Prior: {with_prior}, Train TE: {train_text_encoder}, n_images: {n_images or 'all'}, steps: {max_steps}, lr: {lr}")
    print(f"  Mid-training preview steps: {sorted(preview_steps)}")
    print(f"{'='*60}")

    while global_step < max_steps:
        for batch in loader:
            if global_step >= max_steps:
                break

            inst_pixels = batch["instance_pixel_values"].to(device, dtype=torch.float32)
            inst_ids = batch["instance_input_ids"].to(device)

            if with_prior:
                cls_pixels = batch["class_pixel_values"].to(device, dtype=torch.float32)
                cls_ids = batch["class_input_ids"].to(device)
                pixel_values = torch.cat([inst_pixels, cls_pixels], dim=0)
                input_ids = torch.cat([inst_ids, cls_ids], dim=0)
            else:
                pixel_values = inst_pixels
                input_ids = inst_ids

            with torch.no_grad():
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=device).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            encoder_hidden_states = text_encoder(input_ids)[0]
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

            if with_prior:
                pred_inst, pred_cls = model_pred.chunk(2)
                noise_inst, noise_cls = noise.chunk(2)
                instance_loss = F.mse_loss(pred_inst.float(), noise_inst.float(), reduction="mean")
                prior_loss = F.mse_loss(pred_cls.float(), noise_cls.float(), reduction="mean")
                loss = instance_loss + CFG["lambda_prior"] * prior_loss
                prior_value = float(prior_loss.detach().cpu())
            else:
                instance_loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")
                loss = instance_loss
                prior_value = np.nan

            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, CFG["max_grad_norm"])
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            if global_step % 25 == 0 or global_step == 1 or global_step == max_steps:
                row = {
                    "step": global_step,
                    "loss": float(loss.detach().cpu()),
                    "instance_loss": float(instance_loss.detach().cpu()),
                    "prior_loss": prior_value,
                }
                loss_rows.append(row)
                if global_step % 100 == 0 or global_step == 1:
                    print(f"  step {global_step}: loss={row['loss']:.4f} inst={row['instance_loss']:.4f} prior={prior_value:.4f}")

            if global_step in preview_steps:
                unet.eval()
                text_encoder.eval()
                preview_pipe = build_inference_pipe(tokenizer, text_encoder, vae, unet, use_fp16=False)
                save_training_preview(preview_pipe, run_dir, subject_name, class_noun, is_living, global_step)
                del preview_pipe
                free_memory()
                unet.train()
                if train_text_encoder:
                    text_encoder.train()

    loss_df = pd.DataFrame(loss_rows)
    loss_df.to_csv(run_dir / "loss_history.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(loss_df["step"], loss_df["loss"], label="total", linewidth=1.5)
    ax.plot(loss_df["step"], loss_df["instance_loss"], label="instance", linewidth=1, alpha=0.8)
    if with_prior:
        ax.plot(loss_df["step"], loss_df["prior_loss"], label="prior", linewidth=1, alpha=0.8)
    ax.set_xlabel("Step")
    ax.set_ylabel("MSE Loss")
    ax.set_title(run_dir.name)
    ax.legend()
    plt.tight_layout()
    fig.savefig(run_dir / "loss_curve.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

    pipe = build_inference_pipe(tokenizer, text_encoder, vae, unet)
    return pipe

print("✓ train_dreambooth() defined — includes mid-training preview at step 400")


✓ train_dreambooth() defined — includes mid-training preview at step 400


In [36]:
@torch.no_grad()
def generate_images(pipe, prompts, out_folder, images_per_prompt=4, prefix="img"):
    """Generate images for a list of prompts, save to disk, return paths grouped by prompt."""
    out_folder = Path(out_folder)
    out_folder.mkdir(parents=True, exist_ok=True)
    all_paths = []
    generator = torch.Generator(device=get_device()).manual_seed(CFG["seed"])

    print(f"Generating {len(prompts)} prompts × {images_per_prompt} images = {len(prompts) * images_per_prompt} images")
    print("Saving to:", out_folder)

    for p_idx, prompt in enumerate(tqdm(prompts, desc=f"Generating {prefix} prompts")):
        prompt_paths = []
        for i_idx in range(images_per_prompt):
            img = pipe(
                prompt,
                num_inference_steps=CFG["sample_steps"],
                guidance_scale=CFG["guidance_scale"],
                generator=generator,
            ).images[0]
            path = out_folder / f"{prefix}_p{p_idx:02d}_i{i_idx}.png"
            img.save(path)
            prompt_paths.append(str(path))
        all_paths.append(prompt_paths)
    return all_paths


def generate_run_outputs(pipe, run_dir, subject_name, class_noun, is_living, identifiers=None):
    """Generate eval grid and 4 drift images. Returns dict of paths."""
    identifiers = identifiers or [CFG["identifier"]]
    run_dir = Path(run_dir)

    all_generations = []
    all_prompts = []

    for ident in identifiers:
        prompts = get_eval_prompts(subject_name, class_noun, is_living, identifier=ident)
        gen_dir = run_dir / "generated"
        generations = generate_images(
            pipe, prompts, gen_dir,
            images_per_prompt=CFG["images_per_prompt"],
            prefix=ident,
        )
        all_generations.extend(generations)
        all_prompts.extend(prompts)

        image_rows = []
        for prompt, paths in zip(prompts, generations):
            image_rows.append([Image.open(p) for p in paths])
        make_labeled_grid(
            image_rows, prompts,
            run_dir / "grid.png",
            title=f"{run_dir.parent.name}/{run_dir.name}",
        )
        print(f"Saved rich prompt grid to {run_dir / 'grid.png'}")

    # Drift images: class prompt with no identifier
    drift_prompt = class_prompt(class_noun)
    drift_dir = run_dir / "drift"
    drift_paths = generate_images(
        pipe, [drift_prompt], drift_dir,
        images_per_prompt=4, prefix="drift",
    )[0]
    for i, p in enumerate(drift_paths):
        shutil.copy2(p, run_dir / f"drift_{i}.png")
    print(f"Saved drift images for prompt: '{drift_prompt}'")

    return {
        "generations": all_generations,
        "prompts": all_prompts,
        "drift_images": [str(p) for p in drift_paths],
    }

print("✓ Inference / image-generation helpers defined")


✓ Inference / image-generation helpers defined


In [37]:
def append_metrics_row(row):
    """Append one metrics row to CSV immediately (crash-resilient)."""
    row = dict(row)
    row["timestamp"] = datetime.now().isoformat(timespec="seconds")
    df = pd.DataFrame([row])
    header = not METRICS_CSV.exists()
    df.to_csv(METRICS_CSV, mode="a", header=header, index=False)
    print(f"  -> appended metrics: {row['run_key']}")


def cosine_mean(a, b):
    a = F.normalize(a, dim=-1)
    b = F.normalize(b, dim=-1)
    return float((a @ b.T).mean().detach().cpu())


def load_metric_models():
    device = get_device()
    dino = torch.hub.load(CFG["dino_repo"], CFG["dino_model"]).to(device).eval()
    clip_model = CLIPModel.from_pretrained(CFG["clip_model_name"]).to(device).eval()
    clip_processor = CLIPProcessor.from_pretrained(CFG["clip_model_name"])
    lpips_model = lpips.LPIPS(net="alex").to(device).eval()
    return dino, clip_model, clip_processor, lpips_model


def dino_features(dino, paths, batch_size=16):
    device = get_device()
    tfm = transforms.Compose([
        transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    feats = []
    for i in range(0, len(paths), batch_size):
        imgs = [tfm(Image.open(p).convert("RGB")) for p in paths[i:i+batch_size]]
        x = torch.stack(imgs).to(device)
        with torch.no_grad():
            feats.append(dino(x).detach().cpu())
    return torch.cat(feats, dim=0)


def clip_image_features(clip_model, clip_processor, paths, batch_size=16):
    device = get_device()
    feats = []
    for i in range(0, len(paths), batch_size):
        imgs = [Image.open(p).convert("RGB") for p in paths[i:i+batch_size]]
        inputs = clip_processor(images=imgs, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)
        with torch.no_grad():
            vision_out = clip_model.vision_model(pixel_values=pixel_values, return_dict=True)
            pooled = vision_out.pooler_output
            f = clip_model.visual_projection(pooled)
            f = F.normalize(f, dim=-1)
        feats.append(f.detach().cpu())
    return torch.cat(feats, dim=0)


def clip_text_image_score(clip_model, clip_processor, image_paths, prompts, batch_size=16):
    device = get_device()
    scores = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        batch_prompts = prompts[i:i+batch_size]
        imgs = [Image.open(p).convert("RGB") for p in batch_paths]
        inputs = clip_processor(
            text=batch_prompts,
            images=imgs,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        pixel_values = inputs["pixel_values"].to(device)
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)
        with torch.no_grad():
            out = clip_model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True,
            )
            img_f = F.normalize(out.image_embeds, dim=-1)
            txt_f = F.normalize(out.text_embeds, dim=-1)
            batch_scores = (img_f * txt_f).sum(dim=-1)
        scores.extend(batch_scores.detach().cpu().tolist())
    return float(np.mean(scores))


def lpips_diversity(lpips_model, image_paths_by_prompt):
    device = get_device()
    tfm = transforms.Compose([
        transforms.Resize((256, 256), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])
    per_prompt = []
    for paths in image_paths_by_prompt:
        imgs = [tfm(Image.open(p).convert("RGB")).to(device) for p in paths]
        dists = []
        for i in range(len(imgs)):
            for j in range(i + 1, len(imgs)):
                with torch.no_grad():
                    d = lpips_model(imgs[i].unsqueeze(0), imgs[j].unsqueeze(0))
                dists.append(float(d.detach().cpu()))
        if dists:
            per_prompt.append(np.mean(dists))
    return float(np.mean(per_prompt)) if per_prompt else np.nan


def score_run(run_key, result):
    print(f"  Scoring {run_key}...")
    free_memory()
    dino, clip_model, clip_processor, lpips_model = load_metric_models()

    ref_paths = result["reference_images"]
    image_paths_by_prompt = result["generations"]
    gen_paths = [p for row in image_paths_by_prompt for p in row]
    gen_prompts = []
    for prompt, row in zip(result["prompts"], image_paths_by_prompt):
        gen_prompts.extend([prompt] * len(row))
    drift_paths = result["drift_images"]

    ref_dino = dino_features(dino, ref_paths)
    gen_dino = dino_features(dino, gen_paths)
    drift_dino = dino_features(dino, drift_paths)

    ref_clip = clip_image_features(clip_model, clip_processor, ref_paths)
    gen_clip = clip_image_features(clip_model, clip_processor, gen_paths)

    row = {
        "run_key": run_key,
        "subject": result["subject"],
        "run_name": result["run_name"],
        "class_noun": result["class_noun"],
        "n_images": result["n_images"],
        "with_prior": result["with_prior"],
        "train_text_encoder": result["train_text_encoder"],
        "DINO": round(cosine_mean(gen_dino, ref_dino), 4),
        "CLIP_I": round(cosine_mean(gen_clip, ref_clip), 4),
        "CLIP_T": round(clip_text_image_score(clip_model, clip_processor, gen_paths, gen_prompts), 4),
        "PRES": round(cosine_mean(drift_dino, ref_dino), 4),
        "DIV": round(lpips_diversity(lpips_model, image_paths_by_prompt), 4),
    }
    append_metrics_row(row)

    del dino, clip_model, clip_processor, lpips_model
    free_memory()
    return row

print("✓ Metrics defined (robust CLIP + DINO + LPIPS)")


✓ Metrics defined (robust CLIP + DINO + LPIPS)


In [38]:
def write_metadata(run_dir, metadata):
    lines = ["DreamBooth rich-prompt run metadata", "=" * 36]
    for k, v in metadata.items():
        lines.append(f"{k}: {v}")
    lines.append("\nFull config snapshot:")
    lines.append(json.dumps(CFG, indent=2, default=str))
    with open(Path(run_dir) / "metadata.txt", "w") as f:
        f.write("\n".join(lines))


def run_experiment(subject_name, run_name, class_noun=None, is_living=None,
                   n_images=None, with_prior=True, train_text_encoder=False,
                   instance_sources=None, eval_identifiers=None,
                   score=True, return_pipe=False):
    class_noun = class_noun or CLASS_MAP.get(subject_name, "object")
    if is_living is None:
        is_living = class_noun in ["dog", "cat"]
    run_key = f"{subject_name}_{run_name}"
    run_dir = OUT / subject_name / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    if run_key in ALL_RESULTS:
        print(f"⏭ Skipping {run_key} (already done)")
        return ALL_RESULTS[run_key]

    if instance_sources is None:
        instance_sources = [(DATASET_ROOT / subject_name, instance_prompt(class_noun, CFG["identifier"]))]

    reference_images = []
    for folder, _ in instance_sources:
        imgs = list_images(folder)
        if n_images is not None:
            imgs = imgs[:n_images]
        reference_images.extend(imgs)

    metadata = {
        "run_key": run_key,
        "subject": subject_name,
        "run_name": run_name,
        "class_noun": class_noun,
        "n_images": n_images if n_images is not None else "all_available",
        "reference_image_count": len(reference_images),
        "with_prior": with_prior,
        "train_text_encoder": train_text_encoder,
        "learning_rate": CFG["lr_te"] if train_text_encoder else CFG["lr_unet"],
        "instance_sources": [(str(a), b) for a, b in instance_sources],
        "started": datetime.now().isoformat(timespec="seconds"),
    }
    write_metadata(run_dir, metadata)

    pipe = train_dreambooth(
        run_dir=run_dir,
        instance_sources=instance_sources,
        class_noun=class_noun,
        with_prior=with_prior,
        train_text_encoder=train_text_encoder,
        n_images=n_images,
        max_steps=CFG["max_train_steps"],
    )

    output_info = generate_run_outputs(
        pipe=pipe,
        run_dir=run_dir,
        subject_name=subject_name,
        class_noun=class_noun,
        is_living=is_living,
        identifiers=eval_identifiers or [CFG["identifier"]],
    )

    result = {
        "run_key": run_key,
        "subject": subject_name,
        "run_name": run_name,
        "class_noun": class_noun,
        "is_living": is_living,
        "n_images": n_images if n_images is not None else len(reference_images),
        "with_prior": with_prior,
        "train_text_encoder": train_text_encoder,
        "learning_rate": CFG["lr_te"] if train_text_encoder else CFG["lr_unet"],
        "reference_images": [str(p) for p in reference_images],
        "run_dir": str(run_dir),
        **output_info,
    }

    # Save image paths BEFORE metrics scoring.
    # This protects the run if DINO/CLIP/LPIPS scoring crashes.
    ALL_RESULTS[run_key] = result
    save_all_results()
    print(f"  Saved generated image paths before scoring: {run_key}")

    if score:
        try:
            print(f"  Scoring {run_key}...")
            metrics = score_run(run_key, result)
            ALL_RESULTS[run_key]["metrics"] = metrics
            ALL_RESULTS[run_key].update(metrics)
            save_all_results()
        except Exception as e:
            ALL_RESULTS[run_key]["metrics_error"] = repr(e)
            save_all_results()
            print("  Metrics failed, but generated images/results were saved.")
            print("  Error:", repr(e))

    del pipe
    free_memory()
    return result

print("✓ run_experiment() defined")


✓ run_experiment() defined


## 5. Vanilla SD Baseline Generation

In [39]:
def generate_vanilla_baselines():
    seed_everything(CFG["seed"])
    device = get_device()
    pipe = StableDiffusionPipeline.from_pretrained(
        CFG["pretrained_model"],
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(device)
    pipe.set_progress_bar_config(disable=True)

    generator = torch.Generator(device=device).manual_seed(CFG["seed"])
    for subj in CFG["subjects"]:
        name, cn = subj["name"], subj["class_noun"]
        out_dir = VANILLA_ROOT / name
        out_dir.mkdir(parents=True, exist_ok=True)
        if (out_dir / "img_3.png").exists():
            print(f"  {name}: already exists, skipping")
            continue
        prompt = class_prompt(cn)
        for i in range(4):
            img = pipe(prompt, num_inference_steps=CFG["sample_steps"],
                       guidance_scale=CFG["guidance_scale"],
                       generator=generator).images[0]
            img.save(out_dir / f"img_{i}.png")
        print(f"  {name}: saved 4 vanilla images ('{prompt}')")

    del pipe
    free_memory()
    print("✓ Vanilla baselines ready")

generate_vanilla_baselines()


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  dog2: saved 4 vanilla images ('a photo of a dog')
  cat2: saved 4 vanilla images ('a photo of a cat')
  clock: saved 4 vanilla images ('a photo of a clock')
✓ Vanilla baselines ready


## 6. Class Image Generation

In [40]:
def generate_class_images_for_class(class_noun):
    class_dir = CLASS_ROOT / safe_name(class_noun)
    class_dir.mkdir(parents=True, exist_ok=True)
    existing = list_images(class_dir)
    needed = CFG["num_class_images"] - len(existing)
    if needed <= 0:
        print(f"  '{class_noun}': {len(existing)} images exist, skipping")
        return

    print(f"  Generating {needed} class images for '{class_noun}'...")
    device = get_device()
    pipe = StableDiffusionPipeline.from_pretrained(
        CFG["pretrained_model"], torch_dtype=torch.float16,
        safety_checker=None, requires_safety_checker=False,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(device)
    pipe.set_progress_bar_config(disable=True)

    prompt = class_prompt(class_noun)
    count = len(existing)
    generator = torch.Generator(device=device).manual_seed(CFG["seed"])
    batch = CFG["class_gen_batch"]

    while count < CFG["num_class_images"]:
        n = min(batch, CFG["num_class_images"] - count)
        imgs = pipe([prompt] * n, num_inference_steps=CFG["class_gen_steps"],
                    guidance_scale=CFG["guidance_scale"],
                    generator=generator).images
        for img in imgs:
            img.save(class_dir / f"class_{count:04d}.png")
            count += 1

    del pipe
    free_memory()
    print(f"    -> {count} total images in {class_dir}")

# Generate for all unique class nouns
unique_classes = list(set(s["class_noun"] for s in CFG["subjects"]))
print("Class nouns:", unique_classes)
for cn in unique_classes:
    generate_class_images_for_class(cn)
print("✓ Class images ready")


Class nouns: ['dog', 'clock', 'cat']
  Generating 200 class images for 'dog'...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

    -> 200 total images in /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/class_images_final/dog
  Generating 200 class images for 'clock'...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

    -> 200 total images in /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/class_images_final/clock
  Generating 200 class images for 'cat'...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

    -> 200 total images in /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/class_images_final/cat
✓ Class images ready


## 7. Focused Runs: dog2 + cat2 + clock (main + no_prior)

In [41]:
for subj in CFG["subjects"]:
    name = subj["name"]
    cn = subj["class_noun"]
    living = subj["is_living"]

    print("\n" + "=" * 80)
    print("MAIN:", name)
    run_experiment(
        subject_name=name, run_name="main",
        class_noun=cn, is_living=living,
        n_images=None, with_prior=True, train_text_encoder=False,
    )

    print("\n" + "=" * 80)
    print("NO PRIOR:", name)
    run_experiment(
        subject_name=name, run_name="no_prior",
        class_noun=cn, is_living=living,
        n_images=None, with_prior=False, train_text_encoder=False,
    )

print("\n✓ All 6 focused runs complete.")



MAIN: dog2


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]


Training: main
  Prior: True, Train TE: False, n_images: all, steps: 800, lr: 2.5e-06
  Mid-training preview steps: [400]
  step 1: loss=1.4377 inst=0.7398 prior=0.6979
  step 100: loss=0.0892 inst=0.0838 prior=0.0054
  step 200: loss=0.4768 inst=0.2245 prior=0.2523
  step 300: loss=0.0915 inst=0.0214 prior=0.0701
  step 400: loss=0.3777 inst=0.1136 prior=0.2641

Generating mid-training preview at training step 400...


preview step 400:   0%|          | 0/8 [00:00<?, ?it/s]

Saved preview grid: /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/dog2/main/preview_step_400.png
  step 500: loss=0.5489 inst=0.0479 prior=0.5009
  step 600: loss=0.1703 inst=0.1435 prior=0.0268
  step 700: loss=0.4317 inst=0.1551 prior=0.2765
  step 800: loss=0.0765 inst=0.0618 prior=0.0147
Generating 8 prompts × 4 images = 32 images
Saving to: /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/dog2/main/generated


...

Generating drift prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Saved drift images for prompt: 'a photo of a clock'
  Saved generated image paths before scoring: clock_no_prior
  Scoring clock_no_prior...
  Scoring clock_no_prior...


Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
  -> appended metrics: clock_no_prior

✓ All 6 focused runs complete.


## 8. Dog Ablations: n=3, n=4, text encoder unfrozen

In [42]:
if CFG.get("RUN_DOG_ABLATIONS", False):
    head_cfg = next(s for s in CFG["subjects"] if s["name"] == CFG["headline_subject"])
    head_name = head_cfg["name"]

    run_experiment(
        subject_name=head_name, run_name="n3",
        class_noun=head_cfg["class_noun"], is_living=head_cfg["is_living"],
        n_images=3, with_prior=True, train_text_encoder=False,
    )

    run_experiment(
        subject_name=head_name, run_name="n4",
        class_noun=head_cfg["class_noun"], is_living=head_cfg["is_living"],
        n_images=4, with_prior=True, train_text_encoder=False,
    )

    run_experiment(
        subject_name=head_name, run_name="te_unfrozen",
        class_noun=head_cfg["class_noun"], is_living=head_cfg["is_living"],
        n_images=None, with_prior=True, train_text_encoder=True,
    )

    print(f"\n✓ All 3 headline-subject ablations complete for {head_name}.")
else:
    print("Headline-subject ablations skipped. Set CFG['RUN_DOG_ABLATIONS'] = True if you want n3/n4/TE runs.")


Headline-subject ablations skipped. Set CFG['RUN_DOG_ABLATIONS'] = True if you want n3/n4/TE runs.


## 9. Per-Subject Prior Preservation Figures

In [43]:
def make_prior_vs_no_prior(subject):
    subj_dir = OUT / subject
    vanilla = sorted((VANILLA_ROOT / subject).glob("img_*.png"))[:4]
    no_prior = [Path(p) for p in ALL_RESULTS[f"{subject}_no_prior"]["drift_images"]]
    main = [Path(p) for p in ALL_RESULTS[f"{subject}_main"]["drift_images"]]

    cn = next(s["class_noun"] for s in CFG["subjects"] if s["name"] == subject)
    drift_prompt = f"a photo of a {cn}"

    make_labeled_grid(
        [vanilla, no_prior, main],
        [
            f"Vanilla SD: '{drift_prompt}'",
            f"No prior pres. drift",
            f"With prior pres. drift",
        ],
        subj_dir / "prior_vs_no_prior.png",
    )
    print(f"  Saved {subj_dir / 'prior_vs_no_prior.png'}")

for subj in CFG["subjects"]:
    make_prior_vs_no_prior(subj["name"])

print("✓ All prior_vs_no_prior figures saved.")


  Saved /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/dog2/prior_vs_no_prior.png
  Saved /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/cat2/prior_vs_no_prior.png
  Saved /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/clock/prior_vs_no_prior.png
✓ All prior_vs_no_prior figures saved.


## 10. Dog Visual Comparison Figures

In [44]:
head_dir = OUT / CFG["headline_subject"]

if CFG.get("RUN_DOG_ABLATIONS", False):
    n_sweep_path = head_dir / "n_sweep.png"
    stitch_existing_grids(
        [head_dir / "n3" / "grid.png",
         head_dir / "n4" / "grid.png",
         head_dir / "main" / "grid.png"],
        ["n=3", "n=4", "n=5/all"],
        n_sweep_path,
    )

    te_compare_path = head_dir / "te_compare.png"
    stitch_existing_grids(
        [head_dir / "main" / "grid.png",
         head_dir / "te_unfrozen" / "grid.png"],
        ["TE Frozen (main)", "TE Trained"],
        te_compare_path,
    )

    print("✓ Headline-subject ablation figures saved.")
else:
    print("Headline-subject ablation figures skipped because RUN_DOG_ABLATIONS is False.")


Headline-subject ablation figures skipped because RUN_DOG_ABLATIONS is False.


## 11. Global Metrics Table

In [45]:
def make_metrics_table():
    if not METRICS_CSV.exists():
        print(f"No metrics CSV found yet: {METRICS_CSV}")
        return
    df = pd.read_csv(METRICS_CSV)
    metric_cols = ["DINO", "CLIP_I", "CLIP_T", "PRES", "DIV"]
    show_cols = ["subject", "run_name", "n_images", "with_prior",
                 "train_text_encoder"] + metric_cols
    table_df = df[[c for c in show_cols if c in df.columns]].copy()
    for c in metric_cols:
        if c in table_df.columns:
            table_df[c] = table_df[c].map(lambda x: f"{x:.4f}" if pd.notna(x) else "")

    print(table_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(14, max(3, len(table_df) * 0.45 + 1)))
    ax.axis("off")
    table = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc="center", loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.2, 1.5)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor("#4472C4")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#D9E2F3")
    plt.title("DreamBooth Reproduction — All Metrics", fontsize=14, pad=20)
    out = OUT / "metrics_table.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n✓ Metrics table saved to {out}")

make_metrics_table()


subject run_name  n_images  with_prior  train_text_encoder   DINO CLIP_I CLIP_T   PRES    DIV
   dog2     main         6        True               False 0.4716 0.7587 0.3260 0.0480 0.6839
   dog2 no_prior         6       False               False 0.7653 0.8181 0.2934 0.8486 0.6111
   cat2     main         5        True               False 0.6966 0.8113 0.3001 0.4302 0.6950
   cat2 no_prior         5       False               False 0.7657 0.8323 0.2941 0.8243 0.6515
  clock     main         6        True               False 0.4371 0.7765 0.3005 0.3621 0.6945
  clock no_prior         6       False               False 0.5031 0.8079 0.2881 0.5273 0.6937

✓ Metrics table saved to /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/metrics_table.png


## 12. Final Summary

In [46]:
print("Final output root:", OUT)

print("\nGlobal files:")
for p in [METRICS_CSV, OUT / "metrics_table.png", RESULTS_PKL]:
    print(" ", p)

print("\nPer-subject figures:")
for subj in CFG["subjects"]:
    name = subj["name"]
    print(f"\n  {name}")
    print(f"    step-400 preview: {OUT / name / 'main' / 'preview_step_400.png'}")
    print(f"    main grid step 800: {OUT / name / 'main' / 'grid.png'}")
    print(f"    no_prior step-400: {OUT / name / 'no_prior' / 'preview_step_400.png'}")
    print(f"    no_prior grid:     {OUT / name / 'no_prior' / 'grid.png'}")
    print(f"    prior_vs_no:       {OUT / name / 'prior_vs_no_prior.png'}")

if CFG.get("RUN_DOG_ABLATIONS", False):
    head = CFG["headline_subject"]
    print(f"\n  {head} ablations:")
    print(f"    n_sweep:         {OUT / head / 'n_sweep.png'}")
    print(f"    te_compare:      {OUT / head / 'te_compare.png'}")
else:
    print("\nHeadline-subject ablations were skipped for this focused run.")

print(f"\nTotal runs completed: {len(ALL_RESULTS)}")
print("\n✓ All done! Use preview_step_400.png vs grid.png to compare training progress.")


Final output root: /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview

Global files:
  /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/all_metrics.csv
  /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/metrics_table.png
  /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/all_results.pkl

Per-subject figures:

  dog2
    step-400 preview: /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/dog2/main/preview_step_400.png
    main grid step 800: /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/dog2/main/grid.png
    no_prior step-400: /content/drive/MyDrive/FinalProjectDL/final_run_richprompts_final2_dog2_cat2_clock_lr25_fullpreview/dog2/no_prior/preview_step_400.png
    no_prior grid:     /co

## 13. Optional Inline Figure Viewer

In [47]:
from IPython.display import display, Markdown, Image as IPyImage

def show_png(path, width=1100):
    path = Path(path)
    if path.exists():
        display(Markdown(f"### `{path.relative_to(OUT)}`"))
        display(IPyImage(filename=str(path), width=width))
    else:
        print("Missing:", path)

for subj in CFG["subjects"]:
    name = subj["name"]
    display(Markdown(f"# {name}: training progress + final grids"))
    show_png(OUT / name / "main" / "preview_step_400.png", width=900)
    show_png(OUT / name / "main" / "grid.png", width=1100)
    show_png(OUT / name / "no_prior" / "preview_step_400.png", width=900)
    show_png(OUT / name / "no_prior" / "grid.png", width=1100)
    show_png(OUT / name / "prior_vs_no_prior.png", width=950)

head = CFG["headline_subject"]
if (OUT / head / "n_sweep.png").exists():
    display(Markdown(f"# {head} ablations"))
    show_png(OUT / head / "n_sweep.png", width=1200)
    show_png(OUT / head / "te_compare.png", width=1200)


Output hidden; open in https://colab.research.google.com to view.